In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/sjk/DISSERTATION/.streamlit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from huggingface_hub import login

login("hf_")

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_name):
    """Call this ONCE at the start of your script."""
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    print("Loading model weights (this should only happen once)...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16, # Note: fixed 'dtype' to 'torch_dtype'
        device_map="auto"
    )
    return model, tokenizer


In [4]:
# --- HOW TO RUN IT ---

MODEL_NAME = "Qwen/Qwen3.5-4B"

# 1. Load once
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)

Loading tokenizer...
Loading model weights (this should only happen once)...


Fetching 2 files: 100%|██████████| 2/2 [29:11<00:00, 875.64s/it] 
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 4329.43it/s]


In [5]:
def generate_plan(model, tokenizer, prompt, max_new_tokens, temperature):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 1. Generate the tokenized inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # 2. Extract the actual input_ids tensor and move IT to the GPU
    input_ids = inputs.input_ids.to(model.device)

    # 3. Pass input_ids to the model
    with torch.no_grad():
        outputs = model.generate(
            input_ids,                  # <-- Pass the tensor here
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature
        )

    # 4. Slice out the newly generated tokens using input_ids
    response = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:], # <-- Use input_ids here too
        skip_special_tokens=True
    )

    return response

In [ ]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 3072, 0.3 )

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 2048, 0.3 )

In [ ]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 1024, 0.3 )

In [ ]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 3072, 0.7 )

In [ ]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 2048, 0.7 )

In [ ]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 1024, 0.7 )

In [7]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 20, 0.3 )

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: Technical'

In [8]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 200, 0.3 )

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: Technical joke.\n    *   Goal: Tell a joke that appeals to someone with technical knowledge (programming, IT, engineering, etc.).\n    *   Tone: Humorous, lighthearted, appropriate.\n\n2.  **Brainstorm Technical Jokes:**\n    *   *Programming:* Null pointer, infinite loops, recursion, bugs, spaghetti code, "Hello World", etc.\n    *   *Hardware:* RAM, CPU, hard drives, servers, cables.\n    *   *Networking:* Ping, DNS, IP addresses, firewalls.\n    *   *General Tech:* IT support, coffee, debugging.\n\n3.  **Select a Strong Candidate:**\n    *   *Option 1 (The "Why" joke):* Why did the developer go broke? Because he used up all his cache. (A bit weak).\n    *'

In [9]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 2000, 0.3 )

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: Technical joke.\n    *   Goal: Tell a joke that appeals to someone with technical knowledge (programming, engineering, IT, etc.).\n    *   Tone: Humorous, lighthearted, clever.\n\n2.  **Brainstorming Technical Jokes:**\n    *   *Programming:* Bugs, loops, variables, recursion, debugging, code golf, etc.\n    *   *Hardware:* RAM, CPU, hard drives, cables, etc.\n    *   *Networking:* DNS, ping, IP, latency, etc.\n    *   *Math/Science:* Pi, infinity, etc.\n\n3.  **Selecting a Good Candidate:**\n    *   *Option 1 (Programming):* Why do programmers prefer dark mode? Because light attracts bugs. (A bit cliché).\n    *   *Option 2 (Programming):* What\'s a programmer\'s favorite hangout place? The terminal. (Okay, but maybe too simple).\n    *   *Option 3 (Programming):* Why did the developer go broke? Because he used up all his cache. (Classic).\n    *   *Option 4 (Hardware/Network):* Why don\'t programmers like nature? Too 

In [10]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 20000, 0.3 )

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: Technical joke.\n    *   Goal: Tell a joke that is funny to someone with technical knowledge (programming, IT, engineering, etc.).\n    *   Tone: Lighthearted, witty, appropriate.\n\n2.  **Brainstorming Technical Jokes:**\n    *   *Programming:* `null` vs `undefined`, recursion, infinite loops, bugs, code golf.\n    *   *Hardware:* RAM, CPU, hard drives, power supply.\n    *   *Networking:* Ping, DNS, latency, firewalls.\n    *   *General Tech:* IT support, debugging, version control.\n\n3.  **Selecting a Joke:**\n    *   *Option 1 (Binary):* Why did the computer go to the doctor? Because it had a virus. (Too cliché)\n    *   *Option 2 (SQL):* Why do programmers prefer dark mode? Because light attracts bugs. (A bit old)\n    *   *Option 3 (Python):* What do you call a sleeping Python? A napthone. (Pun-based)\n    *   *Option 4 (General/Classic):* The one about the programmer\'s definition of "bug".\n    *   *Option 5 (T

In [ ]:
generate_plan(model, tokenizer, "Tell me a technical joke." , 200000, 0.3 )

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


: 

In [ ]:
def generate_plan(model_name, prompt, max_new_tokens, temperature):

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
                    model_name
                )
    
    
    # Load the model
    model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.bfloat16,
                device_map= "auto"
            )
    
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # if not isinstance(inputs, torch.Tensor):
    #     inputs = torch.tensor(inputs)

    if hasattr(inputs, "input_ids"):
        inputs = inputs.input_ids

    inputs = inputs.to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature= temperature
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return response 



In [ ]:
generate_plan("Qwen/Qwen3.5-4B", "Tell me a technical joke." , 0.3 )


In [ ]:
generate_plan("Qwen/Qwen3.5-4B", "Tell me a technical joke." , 0.3 )


In [ ]:
generate_plan("Qwen/Qwen3-4B-Thinking-2507", "Tell me a technical joke." , 0.7 )


In [ ]:
generate_plan("Qwen/Qwen3-4B-Thinking-2507", "Tell me a technical joke." , 0.3 )


In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "Tell me a technical joke." , 0.7 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "Tell me a technical joke." , 0.3 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "Tell me about Iron Man" , 0.3 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "Tell me about Iron Man" , 0.7 )

In [ ]:
model_name = "google/gemma-4-E4B-it"

def generate_plan():

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
                    model_name
                )
    
    
    # Load the model
    model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.bfloat16,
                device_map="cpu"
            )
    
    messages = [
        {
            "role": "user",
            "content": "Tell me a technical joke."
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # if not isinstance(inputs, torch.Tensor):
    #     inputs = torch.tensor(inputs)

    if hasattr(inputs, "input_ids"):
        inputs = inputs.input_ids

    inputs = inputs.to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return response 



In [ ]:
generate_plan()

In [ ]:
generate_plan()

In [ ]:
def generate_plan(model_name, device_map, prompt, temperature):

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
                    model_name
                )
    
    
    # Load the model
    model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.bfloat16,
                device_map= device_map
            )
    
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # if not isinstance(inputs, torch.Tensor):
    #     inputs = torch.tensor(inputs)

    if hasattr(inputs, "input_ids"):
        inputs = inputs.input_ids

    inputs = inputs.to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature= temperature
    )

    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:],
        skip_special_tokens=True
    )

    return response 



In [ ]:
generate_plan("google/gemma-4-E4B-it", "auto", "Tell me a technical joke." , 0.7 )

In [ ]:
generate_plan("google/gemma-4-E4B-it", "cuda", "Tell me a technical joke." , 0.7 )

In [ ]:
generate_plan("google/gemma-4-E4B-it", "cpu", "Tell me a technical joke." , 0.7 )

In [ ]:
generate_plan("google/gemma-4-E4B-it", "auto", "Tell me a technical joke." , 0.3 )

In [ ]:
generate_plan("google/gemma-4-E4B-it", "cuda", "Tell me a technical joke." , 0.3 )

In [ ]:
generate_plan("google/gemma-4-E4B-it", "cpu", "Tell me a technical joke." , 0.3 )

#

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "auto", "Tell me a technical joke." , 0.7 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "cuda", "Tell me a technical joke." , 0.7 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "cpu", "Tell me a technical joke." , 0.7 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "auto", "Tell me a technical joke." , 0.3 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "cuda", "Tell me a technical joke." , 0.3 )

In [ ]:
generate_plan("meta-llama/Llama-3.1-8B-Instruct", "cpu", "Tell me a technical joke." , 0.3 )

In [ ]:
import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from huggingface_hub import login

login("hf_")


def generate_plan(
    model_name: str,
    prompt: str,
    temperature: float = 0.7,
    max_new_tokens: int = 200,
    device_map: str = "auto",
    quantization: str = "4bit",
):
    """
    Generic Hugging Face inference function.

    Parameters
    ----------
    model_name : HuggingFace model name
    prompt : User prompt
    temperature : Sampling temperature
    max_new_tokens : Maximum generated tokens
    device_map : auto | cuda | cpu
    quantization : none | 4bit | 8bit
    """

    # ------------------------------------------------------------------
    # Clear GPU memory
    # ------------------------------------------------------------------

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ------------------------------------------------------------------
    # Tokenizer
    # ------------------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Some models don't define a pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ------------------------------------------------------------------
    # Quantization Configuration
    # ------------------------------------------------------------------

    quantization_config = None

    if quantization == "4bit":

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    elif quantization == "8bit":

        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True
        )

    # ------------------------------------------------------------------
    # Load Model
    # ------------------------------------------------------------------

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=device_map,
        torch_dtype=torch.bfloat16,
        quantization_config=quantization_config,
        low_cpu_mem_usage=True,
    )

    model.eval()

    # ------------------------------------------------------------------
    # Create Chat Prompt
    # ------------------------------------------------------------------

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    # ------------------------------------------------------------------
    # Generate
    # ------------------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )

    # ------------------------------------------------------------------
    # Decode only generated tokens
    # ------------------------------------------------------------------

    generated_tokens = outputs[0][inputs.shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return response.strip()

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import transformers
import accelerate

print(transformers.__version__)
print(accelerate.__version__)

import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

response = generate_plan(
    model_name="google/gemma-3-4b-it",
    prompt="Tell me a technical joke.",
    quantization="4bit"
)

print(response)

In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U transformers accelerate